# C10 prereg34：OGS 棋譜的 P1 特徵（Colab）

**不用改任何程式。** 開之前先在「執行階段 → 變更執行階段類型」選 GPU（Claude 指定的型號）和**執行階段版本 2026.07**，再按「執行階段 → 全部執行」。

它做的事：
1. 掛上雲端硬碟，核對 `C10/gpu_fallback/C10_gpu_kit_v1.zip`（10 月演練時上傳的套件，含 KataGo 權重）和 `C10/p34/C10_p34_bundle.zip` 的 SHA-256。
2. 印出 `pip list | grep -iE '^torch |nvidia-cudnn'`；torch 不是 cu128 版就停（表示執行階段版本沒選 2026.07）。
3. 先用 400 盤舊棋（接縫 G34）和家裡 4090 的結果比對，沒過就停。
4. 再算 OGS 棋譜（每 200 盤存一次到雲端硬碟；斷線後重按「全部執行」會接著算）。
5. 全部算完，在 `C10/p34/` 產生 `C10_p34_results_<GPU>.zip`，下載給 Claude。

任何一格出現紅字或 `STOP`，就停下來，把最後 20 行截圖給 Claude。

In [ ]:
# 1. 雲端硬碟、GPU、兩個 zip（不用改）
import glob, hashlib, json, os, shutil, subprocess, sys, time, zipfile
KIT_ZIP = '/content/drive/MyDrive/C10/gpu_fallback/C10_gpu_kit_v1.zip'
KIT_ZIP_SHA256 = '98b6d1e745b0a6b4dd26076e4640b36bddeadd2c1ae730cdf9d60e566d09421c'
KIT_ID = 'g11-kit-v1-de2a1ffa6727'
BUNDLE_ZIP = '/content/drive/MyDrive/C10/p34/C10_p34_bundle.zip'
BUNDLE_SHA256 = 'a71b68aa86de93a90e873876564eee4f14babf42348feebd32907daec7f40355'
P34_DRIVE = '/content/drive/MyDrive/C10/p34'
TORCH_TAG = '+cu128'
WORK = '/content/c10'
OUT = os.path.join(P34_DRIVE, 'out')

def stop(msg):
    raise RuntimeError('STOP: ' + msg)

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''):
            h.update(b)
    return h.hexdigest()

from google.colab import drive
drive.mount('/content/drive')
smi = subprocess.run(['nvidia-smi', '--query-gpu=name,driver_version,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True)
if smi.returncode != 0 or not smi.stdout.strip():
    stop('這個執行階段沒有 GPU：執行階段 → 變更執行階段類型 → 選 Claude 指定的 GPU，再全部執行')
print('GPU:', smi.stdout.strip())
if not os.path.exists(BUNDLE_ZIP):
    # uploaded in pieces (<10 MB each): join them locally; the SHA-256 check below guards the result
    parts = sorted(glob.glob(BUNDLE_ZIP + '.part*'))
    if parts:
        joined = '/content/' + os.path.basename(BUNDLE_ZIP)
        with open(joined, 'wb') as fo:
            for p in parts:
                with open(p, 'rb') as fi:
                    shutil.copyfileobj(fi, fo)
        print('bundle 由', len(parts), '段接回')
        BUNDLE_ZIP = joined
for z in (KIT_ZIP, BUNDLE_ZIP):
    if not os.path.exists(z):
        stop(f'找不到 {z}')
t0 = time.time()
if sha256_file(KIT_ZIP) != KIT_ZIP_SHA256:
    stop('C10_gpu_kit_v1.zip 的 SHA-256 不對（不要重新上傳，截圖給 Claude）')
if sha256_file(BUNDLE_ZIP) != BUNDLE_SHA256:
    stop('C10_p34_bundle.zip 的 SHA-256 不對（重新上傳，或問 Claude）')
print(f'兩個 zip OK（{time.time() - t0:.0f} 秒）')

In [ ]:
# 2. 執行環境：torch 一定要是 cu128 版（執行階段版本 2026.07）（不用改）
pl = subprocess.run("pip list 2>/dev/null | grep -iE '^torch |nvidia-cudnn'", shell=True, capture_output=True, text=True)
print(pl.stdout)
probe = subprocess.run([sys.executable, '-c', 'import json, sys, torch, numpy; print(json.dumps({"python": sys.version.split()[0], '
                        '"torch": torch.__version__, "numpy": numpy.__version__, "cuda": torch.cuda.is_available(), '
                        '"cudnn": torch.backends.cudnn.version(), "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}))'],
                       capture_output=True, text=True)
print(probe.stdout, probe.stderr[-1500:])
if probe.returncode != 0:
    stop('torch 無法載入')
ENV = json.loads(probe.stdout.strip().splitlines()[-1])
torch_lines = [l for l in pl.stdout.splitlines() if l.lower().startswith('torch ')]
if not ENV['torch'].endswith(TORCH_TAG) or not torch_lines or TORCH_TAG not in torch_lines[0]:
    stop(f'torch 是 {ENV["torch"]}，不是 cu128 版：中斷連線並刪除執行階段 → 變更執行階段類型 → 執行階段版本選 2026.07 → 全部執行')
if not ENV['cuda']:
    stop('CUDA 不能用')
KEY = {k: str(ENV[k]) for k in ('torch', 'cudnn', 'gpu', 'python', 'numpy')}
GPU_SLUG = ''.join(c if c.isalnum() else '-' for c in ENV['gpu'].lower()).strip('-')
print('環境 OK：', KEY)

In [ ]:
# 3. 解開套件和 p34 bundle，逐檔核對（不用改；約 1–2 分鐘）
shutil.rmtree(WORK, ignore_errors=True)
os.makedirs(WORK)
for z in (KIT_ZIP, BUNDLE_ZIP):
    with zipfile.ZipFile(z) as zf:
        zf.extractall(WORK)
KIT = os.path.join(WORK, 'c10kit')
with open(os.path.join(KIT, 'KIT_MANIFEST.json'), encoding='utf-8') as f:
    if json.load(f)['kit_id'] != KIT_ID:
        stop('套件版本不對')
with open(os.path.join(KIT, 'P34_BUNDLE_MANIFEST.json'), encoding='utf-8') as f:
    BM = json.load(f)
if BM['kit_id'] != KIT_ID or BM['kit_zip_sha256'] != KIT_ZIP_SHA256:
    stop('p34 bundle 是給另一份套件做的')
bad = [rel for rel, want in BM['files'].items() if sha256_file(os.path.join(KIT, *rel.split('/'))) != want]
if bad:
    stop(f'p34 bundle 檔案不符：{bad}')
env = dict(os.environ, PYTHONDONTWRITEBYTECODE='1')
for k in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'PYTHONUTF8'):
    env.pop(k, None)
v = subprocess.run([sys.executable, os.path.join(KIT, 'colab_gpu', 'gpu_fallback_core.py'), 'verify', '--kit', KIT],
                   capture_output=True, text=True, env=env)
print(v.stdout[-2000:], v.stderr[-2000:])
if v.returncode != 0:
    stop('套件核對失敗（含 KataGo 權重的 SHA-256）')
os.makedirs(OUT, exist_ok=True)
env_path = os.path.join(OUT, 'ENV.json')
if os.path.exists(env_path):          # resume only on the same GPU / torch / cuDNN / python / numpy
    with open(env_path, encoding='utf-8') as f:
        first = json.load(f)
    if first['key'] != KEY or first['bundle_sha256'] != BUNDLE_SHA256:
        stop(f'和第一次開跑的環境不同：{first["key"]} vs {KEY}。不要換 GPU 續跑，截圖給 Claude')
else:
    with open(env_path, 'w', encoding='utf-8') as f:
        json.dump({'key': KEY, 'environment': ENV, 'bundle_sha256': BUNDLE_SHA256, 'kit_id': KIT_ID,
                   'gpu_line': smi.stdout.strip(), 'pip_list': pl.stdout, 'created_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}, f, indent=1)
WORKERS = max(1, min(20, (os.cpu_count() or 2) - 2))
print('核對 OK；結果寫到', OUT, '；workers', WORKERS)

def run_p34(args, log_name):
    cmd = [sys.executable, os.path.join(KIT, 'results', 'prereg34', 'explore', 'p34_extract.py')] + args
    print(' '.join(cmd), flush=True)
    p = subprocess.Popen(cmd, cwd=KIT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    with open(os.path.join(OUT, log_name), 'a', encoding='utf-8') as lf:
        for line in p.stdout:
            print(line, end='', flush=True)
            lf.write(line)
    return p.wait()

In [ ]:
# 4. 接縫 G34：400 盤舊棋在這個 GPU 上重算，和家裡 4090 的結果比（不用改；約 5 分鐘）
rc = run_p34(['run', '--set', 'probe', '--sets', 'data/prereg34/p1/seam_sets.json', '--out-dir', os.path.join(OUT, 'seam'),
              '--workers', str(WORKERS)], 'seam_log.txt')
if rc != 0:
    stop(f'接縫的計算失敗（exit {rc}），截最後 20 行給 Claude')
rc = run_p34(['seam-colab', '--new-dir', os.path.join(OUT, 'seam', 'probe'),
              '--ref', os.path.join(KIT, 'data', 'prereg34', 'p1', 'seam_ref.npz'),
              '--out', os.path.join(OUT, 'seam_colab.json')], 'seam_log.txt')
if rc != 0:
    stop('接縫 G34 沒過：不算 OGS。截圖給 Claude')
print('接縫 G34 過了')

In [ ]:
# 5. OGS 棋譜（不用改；可以續跑。每 200 盤印一行 chunk done，eta_min＝剩幾分鐘）
rc = run_p34(['run', '--set', 'probe', '--sets', 'data/prereg34/p1/sets.json', '--out-dir', os.path.join(OUT, 'ogs'),
              '--workers', str(WORKERS)], 'ogs_log.txt')
if rc != 0:
    stop(f'計算失敗（exit {rc}），截最後 20 行給 Claude')
n_want = (BM['ogs_games'] + 199) // 200
have = sorted(f for f in os.listdir(os.path.join(OUT, 'ogs', 'probe')) if f.startswith('chunk_') and f.endswith('.npz') and '.tmp' not in f)
if len(have) != n_want:
    stop(f'只有 {len(have)}／{n_want} 塊（可能是 STOP 檔）；要繼續就再按一次全部執行')
print('OGS 全部算完：', len(have), '塊')

In [ ]:
# 6. 打包結果（不用改）
res_zip = os.path.join(P34_DRIVE, f'C10_p34_results_{GPU_SLUG}.zip')
tmp_zip = '/content/c10_p34_results.zip'
with zipfile.ZipFile(tmp_zip, 'w', zipfile.ZIP_STORED) as zf:
    for d, _dirs, files in os.walk(OUT):
        for f in sorted(files):
            if '.tmp' in f:
                continue
            p_ = os.path.join(d, f)
            zf.write(p_, os.path.relpath(p_, OUT))
shutil.copyfile(tmp_zip, res_zip)
print('完成：', res_zip, f'{os.path.getsize(res_zip) / 1e6:.1f} MB', 'sha256', sha256_file(res_zip))
print('下載這個檔給 Claude（雲端硬碟 C10 → p34 裡按右鍵 → 下載），然後「執行階段 → 中斷連線並刪除執行階段」。')